Hypothesis testing & PRC of simple de novo simulation.

In [33]:
#imports
import scMPRAforge as scm
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster

In [35]:
#autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
#set up cluster
local=True

if local:
    cluster=LocalCluster(memory_limit='4GB',n_workers=3,threads_per_worker=4,)
    client=Client(cluster)
else:
    from dask_jobqueue import SLURMCluster
    from dask.distributed import Client

    cluster=SLURMCluster(
        cores=4,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=2,#dask workers
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=8:00:00",
            f"--output=slave_%j.out"]
    )

    cluster.scale(jobs=6)

    client = Client(cluster,
            timeout=f"{5*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s"  # Worker heartbeat interval
        )


In [37]:
#load sim & hypo test
data_root="/gpfs/gibbs/pi/reilly/tabula_data"

hypothesis_set=scm.HypothesisSet.from_tsv(f"{data_root}/simulated/activity_de_novo_hypotheses.tsv")

In [39]:
de_novo_sim=None
first_time=False
if first_time:
    de_novo_sim=scm.de_novo_simulation.load(client,data_root,"simulated/activity_de_novo")
    de_novo_sim._test_all_replicates(client=client,
        hypothesis_set=hypothesis_set,
        test="wald")
    de_novo_sim.save(data_root,"simulated/activity_de_novo_and_results")
else:
    de_novo_sim=scm.de_novo_simulation.load(client,data_root,"simulated/activity_de_novo_and_results")

In [41]:
de_novo_sim._test_replicate(client,hypothesis_set,test="mwu",index=0)


TokenizationError: Object <scMPRAforge.core.HypothesisSet object at 0x148f42d4b340> cannot be deterministically hashed. This likely indicates that the object cannot be serialized deterministically.

In [28]:
temp_data=de_novo_sim.simulated_scMPRA[0].result()
runner = scm.HypothesisTester("mwu", alternative="two-sided")  # can override defaults here
mwu_results = runner.run(hypothesis_set, temp_data, client=client)

In [31]:
mwu_results

In [11]:
de_novo_sim.results

{'wald': [<Future: finished, type: scMPRAforge.core.ResultSet, key: lambda-5136abdb054b446788337fd7d30b4d8a>],
 'mwu': [<Future: finished, type: NoneType, key: _test_helper-cec44b457b2b61182d3469385397b3b8>]}

In [15]:
de_novo_sim.save(data_root,"simulated/activity_de_novo_and_results")

2025-09-18 18:27:58,824 - distributed.worker - ERROR - Compute Failed
Key:       ('read-csv-892ce638d60aa998daf0eab8edb3027b', 0)
State:     executing
Task:  <Task ('read-csv-892ce638d60aa998daf0eab8edb3027b', 0) _read_csv(..., ...)>
Exception: "FileNotFoundError(2, 'No such file or directory')"
Traceback: '  File "/home/mcn26/.conda/envs/env_tensorzinb/lib/python3.10/site-packages/dask/bytes/core.py", line 191, in read_block_from_file\n    with copy.copy(lazy_file) as f:\n  File "/home/mcn26/.conda/envs/env_tensorzinb/lib/python3.10/site-packages/fsspec/core.py", line 105, in __enter__\n    f = self.fs.open(self.path, mode=mode)\n  File "/home/mcn26/.conda/envs/env_tensorzinb/lib/python3.10/site-packages/fsspec/spec.py", line 1310, in open\n    f = self._open(\n  File "/home/mcn26/.conda/envs/env_tensorzinb/lib/python3.10/site-packages/fsspec/implementations/local.py", line 201, in _open\n    return LocalFileOpener(path, mode, fs=self, **kwargs)\n  File "/home/mcn26/.conda/envs/env_te

FileNotFoundError: [Errno 2] No such file or directory: '/gpfs/gibbs/pi/reilly/tabula_data/simulated/activity_de_novo_and_results/descriptions/0.tsv.gz'

In [19]:
de_novo_sim.results["mwu"][0].result()

In [7]:
#def _test_replicate(self,client,hypothesis_set,test,index)

In [8]:
de_novo_sim.results['mwu'][0].result()

In [ ]:
de_novo_sim.save(data_root,"simulated/activity_de_novo_and_results")

In [ ]:
def _gt_comp_helper():
    pass

#public method of results object
def compare_to_ground_truth(self,gt):
    """
    Takes a ground truth dataframe.
    Returns a single row frame with lots of useful information about 
    how well the test recapitulated the ground truth. 
    """
    pass

In [ ]:
#public method of de_novo_batch
def summarize_performance(self,hypothesis_set,test):
    """
    Takes a hypothesis set object and a particular test type string and produces a dataframe 
    summarzing performance for all simulated replicates, and saves to `self.performances`.
    Then aggregates all metrics to single values & saves to `self.performance_all`
    This function is a collector and will hang!
    """
    pass
    #Calls _test_all_replicates, then 
    #calls compare_to_ground_truth on each. 
    #aggregates the resuling DFs and 

In [ ]:
#public method of de_novo_batch
def performance_plot(self,plot_type,*args,**kwargs):
    """
    Plots the performance of the executed hypothesis tests.
    
    Requires that you call summarize_performance first. 
    Draws from `self.performance_all`

    args and kwargs are passed to the plotting function to allow 
    the user to modify the appearance of the plot as desired. 
    
    plot type can be one of....
    (for each option show which plotting function the args will be passed to)

    
    """
    pass

In [ ]:
#merge in `comparison` ground truth
merged=results.merge(de_novo_sim.ground_truth,
    left_on=["comparison_CRE","comparison_cell_type"],
    right_on=["cre_id","cell_type"]
)
merged=merged.drop(columns=["cre_id","cell_type"])
merged=merged.rename({"true_mean":"comparison_truth"},axis=1)

#merge in `reference` ground truth
merged=merged.merge(de_novo_sim.ground_truth,
    left_on=["reference_CRE","reference_cell_type"],
    right_on=["cre_id","cell_type"]
)
merged=merged.drop(columns=["cre_id","cell_type"])
merged=merged.rename({"true_mean":"reference_truth"},axis=1)

#ground truth effect size
merged["gt_effect_size"]=merged["comparison_truth"]/merged["reference_truth"]
#ground truth null hypothesis that the CREs are the same : true or false?
merged["gt_null"]=abs(merged["gt_effect_size"]-1)<1e-8

merged["reject_null"]=merged["bh_p"]<0.05

simple_bh_results=pd.crosstab(merged["gt_null"],merged["reject_null"])

#now onto PRC

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score

y_true = (~merged["gt_null"]).astype(int).to_numpy()
p = merged["p_value"].to_numpy()
epsilon = np.finfo(float).tiny
scores = -np.log10(p + epsilon)

# 3) PR curve + AUPRC
prec, rec, _ = precision_recall_curve(y_true, scores)
auprc = average_precision_score(y_true, scores)

# 4) ROC (optional)
fpr, tpr, _ = roc_curve(y_true, scores)
auroc = roc_auc_score(y_true, scores)

# 5) Plot PRC with baseline
pos_rate = y_true.mean()  # prevalence; PRC baseline
plt.figure(figsize=(5,4))
plt.plot(rec, prec, lw=2)
plt.hlines(pos_rate, 0, 1, linestyles="--", label=f"Baseline = {pos_rate:.3f}")
plt.xlim(0, 1); plt.ylim(0, 1)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"PR Curve (AUPRC = {auprc:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

# (Optional) Plot ROC
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, lw=2, label=f"AUROC = {auroc:.3f}")
plt.plot([0,1],[0,1], "--", color="gray")
plt.xlim(0,1); plt.ylim(0,1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()


In [32]:
cluster.close()
client.close()

2025-09-18 18:46:14,626 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:36999' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'lambda-21937a010aa9ef10039ce9aa9d68a9a5'} (stimulus_id='handle-worker-cleanup-1758235574.6265187')
2025-09-18 18:46:14,634 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:44525' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'lambda-9cecd65593c488a0183d49461c6befad'} (stimulus_id='handle-worker-cleanup-1758235574.6346185')
2025-09-18 18:46:14,640 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:36789' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_test_helper-cec44b457b2b61182d3469385397b3b8', 'lambda-5136abdb054b446788337fd7d30b4d8a', 'lambda-32a4af068eb5737701f113109cff4a07'} (stimulus_id='handle-worker-cleanup-1758235574.640762')
